# k-Nearest Neighbors (k-NN) — From First Principles

k-NN is a simple, non-parametric method whose main cost is deferred to prediction time. This notebook implements chunked nearest-neighbor classification, compares it with scikit-learn, and demonstrates why scaling and memory handling matter.

## 1. Definition and intuition

For a query point, k-NN finds the \(k\) closest training samples and predicts from their labels (classification) or target values (regression). It does not learn a compact parameter vector in the way a linear model does; it retains the training examples.

## 2. Mathematics and why it works

For Euclidean distance, \(d(x,z)=\sqrt{\sum_j(x_j-z_j)^2}\). Classification typically uses majority vote among the k nearest labels; distance-weighted voting gives closer examples more influence. The square root can be omitted when ranking points because it is monotonic: squared distances produce the same nearest-neighbor ordering.

Under suitable sampling and regularity conditions, nearest-neighbor methods can approach the Bayes classification error as the sample size grows and \(k\) grows slowly relative to \(n\). In finite data, no universal accuracy guarantee exists: the result depends on feature representation, distance metric, label noise, and the curse of dimensionality.

## 3. Assumptions and failure modes

The chosen distance must reflect meaningful similarity. Features on large scales can dominate; irrelevant dimensions dilute distances; high-dimensional data often make near and far points less distinguishable. k-NN is sensitive to class imbalance, duplicates, noisy labels, and memory constraints. It can model irregular boundaries, but inference cost grows with the training set unless an index or approximate method is used.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

SEED = 42
X, y = make_classification(n_samples=1200, n_features=10, n_informative=6,
                           n_redundant=1, class_sep=1.25, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)
scaler = StandardScaler().fit(X_train)
Xtr = scaler.transform(X_train)
Xte = scaler.transform(X_test)
print('train/test:', Xtr.shape, Xte.shape)

train/test: (900, 10) (300, 10)


## 4. From-scratch chunked k-NN

A naïve broadcast calculation creates an \(m\times n\times d\) difference tensor, which is usually wasteful. The implementation below uses the squared-distance identity to build a batch-by-training distance matrix, then processes query batches to cap temporary memory.

In [2]:
class KNNClassifierScratch:
    def __init__(self, n_neighbors=5, weights='uniform', batch_size=128):
        if n_neighbors < 1 or batch_size < 1:
            raise ValueError('n_neighbors and batch_size must be positive')
        if weights not in {'uniform', 'distance'}:
            raise ValueError("weights must be 'uniform' or 'distance'")
        self.n_neighbors = int(n_neighbors)
        self.weights = weights
        self.batch_size = int(batch_size)

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)
        if X.ndim != 2 or len(X) == 0 or y.ndim != 1 or len(y) != len(X):
            raise ValueError('X must be non-empty 2D; y must be 1D with matching length')
        if not np.isfinite(X).all():
            raise ValueError('X must be finite')
        if self.n_neighbors > len(X):
            raise ValueError('n_neighbors cannot exceed number of training rows')
        self.X_ = X
        self.y_ = y
        self.classes_ = np.unique(y)
        self.X_sq_ = np.einsum('ij,ij->i', X, X)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        if X.ndim != 2 or X.shape[1] != self.X_.shape[1]:
            raise ValueError('X must be 2D and match training feature count')
        if not np.isfinite(X).all():
            raise ValueError('X must be finite')
        predictions = []
        train_sq = self.X_sq_
        for start in range(0, len(X), self.batch_size):
            Q = X[start:start + self.batch_size]
            q_sq = np.einsum('ij,ij->i', Q, Q)
            dist2 = q_sq[:, None] + train_sq[None, :] - 2 * (Q @ self.X_.T)
            np.maximum(dist2, 0, out=dist2)
            idx = np.argpartition(dist2, self.n_neighbors - 1, axis=1)[:, :self.n_neighbors]
            for row, neighbors in zip(dist2, idx):
                labels = self.y_[neighbors]
                if self.weights == 'uniform':
                    counts = np.array([(labels == c).sum() for c in self.classes_])
                else:
                    distances = np.sqrt(row[neighbors])
                    # If any neighbor is exactly coincident, only exact matches vote.
                    exact = distances <= np.finfo(float).eps
                    if exact.any():
                        distances = exact.astype(float)
                    else:
                        distances = 1.0 / np.maximum(distances, 1e-12)
                    counts = np.array([distances[labels == c].sum() for c in self.classes_])
                predictions.append(self.classes_[int(np.argmax(counts))])
        return np.asarray(predictions)

knn = KNNClassifierScratch(n_neighbors=5, batch_size=64).fit(Xtr, y_train)
pred_scratch = knn.predict(Xte)
print('from-scratch test accuracy:', accuracy_score(y_test, pred_scratch))
assert len(pred_scratch) == len(y_test)

from-scratch test accuracy: 0.9433333333333334


## 5. Compare with scikit-learn

Use the same scaled data, metric, k, and uniform vote. Different tie-breaking can make isolated predictions differ, but the overall behavior should be similar.

In [3]:
sk_knn = KNeighborsClassifier(n_neighbors=5, weights='uniform', metric='euclidean', algorithm='brute')
sk_knn.fit(Xtr, y_train)
pred_sklearn = sk_knn.predict(Xte)
print('scikit-learn test accuracy:', accuracy_score(y_test, pred_sklearn))
print(classification_report(y_test, pred_sklearn, zero_division=0))
assert accuracy_score(y_test, pred_scratch) > 0.65
assert accuracy_score(y_test, pred_sklearn) > 0.65

scikit-learn test accuracy: 0.9433333333333334
              precision    recall  f1-score   support

           0       0.95      0.93      0.94       151
           1       0.93      0.95      0.94       149

    accuracy                           0.94       300
   macro avg       0.94      0.94      0.94       300
weighted avg       0.94      0.94      0.94       300



## 6. Explore k and distance weighting

Small k gives a flexible, noisy boundary; larger k smooths it, which may help or hurt depending on the data. Tune k using cross-validation and evaluate once on a held-out test set.

In [4]:
for k in (1, 3, 5, 11, 25):
    model = KNeighborsClassifier(n_neighbors=k, weights='uniform')
    model.fit(Xtr, y_train)
    print(f'k={k:2d} train={model.score(Xtr, y_train):.3f} test={model.score(Xte, y_test):.3f}')
weighted = KNeighborsClassifier(n_neighbors=5, weights='distance').fit(Xtr, y_train)
print('distance-weighted test accuracy:', weighted.score(Xte, y_test))

k= 1 train=1.000 test=0.913
k= 3 train=0.953 test=0.933
k= 5 train=0.940 test=0.943
k=11 train=0.929 test=0.950
k=25 train=0.910 test=0.927
distance-weighted test accuracy: 0.9433333333333334


## 7. Hyperparameters and best practices

- `n_neighbors`: controls the local smoothness of predictions; tune with cross-validation.
- `weights`: uniform or distance-weighted voting.
- `metric` / `p`: define what “near” means; Manhattan and Minkowski distances can behave differently from Euclidean distance.
- `algorithm`: brute force, KD-tree, or Ball-tree; tree indexes often degrade in high dimensions.
- `leaf_size`: speed/memory trade-off for tree-based search.
- Scale numeric features using a train-fitted pipeline; choose imputation and categorical encodings carefully.
- Consider class weighting, balanced metrics, or resampling for imbalanced labels; ordinary neighbor voting may follow the majority class.

## 8. Complexity and scaling

Let \(n\)=training samples, \(d\)=features, \(m\)=queries, \(k\)=neighbors, and \(B\)=query batch size.

| Phase | Brute-force complexity | Space |
|---|---|---|
| Fit | Often just storing data: \(O(nd)\) time | \(O(nd)\) stored training data |
| Predict | \(O(mnd)\) distance work plus neighbor selection | Chunked temporary distances \(O(Bn)\); stored data \(O(nd)\) |

For millions of rows, exact brute-force inference can be expensive. Options include batch/chunk prediction, vectorized distance kernels, approximate nearest-neighbor indexes (e.g. FAISS or HNSW implementations), lower-dimensional embeddings, and GPU-based distance calculations. Approximate search changes the method's exactness and must be evaluated for recall and downstream quality.

## 9. Optional PyTorch/GPU distance calculation

The following helper computes chunked nearest labels using PyTorch. It uses CUDA when available and otherwise runs on CPU. For very large training sets, do not copy all data to a device whose memory cannot hold it; choose the chunk sizes after checking memory.

In [5]:
try:
    import torch
    def knn_predict_torch(X_train, y_train, X_query, k=5, query_batch_size=128, device=None):
        device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        Xt = torch.as_tensor(X_train, dtype=torch.float32, device=device)
        yt = torch.as_tensor(np.asarray(y_train), device=device)
        Xq = torch.as_tensor(X_query, dtype=torch.float32, device=device)
        if k < 1 or k > len(Xt):
            raise ValueError('k must be between 1 and the number of training samples')
        train_sq = (Xt * Xt).sum(dim=1)
        outputs = []
        for start in range(0, len(Xq), query_batch_size):
            Q = Xq[start:start + query_batch_size]
            q_sq = (Q * Q).sum(dim=1, keepdim=True)
            d2 = torch.clamp(q_sq + train_sq.unsqueeze(0) - 2 * Q @ Xt.T, min=0)
            idx = torch.topk(d2, k=k, largest=False, dim=1).indices
            labels = yt[idx]
            # Majority vote works for integer class labels used in this demo.
            pred = torch.mode(labels, dim=1).values
            outputs.append(pred.cpu().numpy())
        return np.concatenate(outputs) if outputs else np.array([]), device
    pred_torch, torch_device = knn_predict_torch(Xtr, y_train.astype(np.int64), Xte, k=5)
    print('PyTorch device:', torch_device, 'test accuracy:', accuracy_score(y_test, pred_torch))
except ImportError:
    print('PyTorch is not installed; skipping optional tensor demonstration.')

PyTorch device: cpu test accuracy: 0.9433333333333334


## 10. Correctness checks and synthesis

k-NN keeps the training set and delays much of the computation until prediction. That makes it conceptually simple and flexible, but its inference-time and memory costs can dominate at scale. The distance metric and feature representation are part of the model, not mere implementation details.

In [6]:
assert np.isfinite(Xtr).all()
try:
    KNNClassifierScratch(n_neighbors=len(Xtr) + 1).fit(Xtr, y_train)
except ValueError:
    pass
else:
    raise AssertionError('Expected k greater than n to fail')
print('All k-NN notebook checks passed.')

All k-NN notebook checks passed.
